# 06 | Cross-Cell Multimodal SOH Forecasting

## Study objective

This notebook tests whether degradation information extracted from EIS, IV and transient-response measurements can improve future state-of-health forecasts for a completely unseen SOFC cell.

The analysis uses leakage-safe nested leave-one-cell-out validation. Model selection is performed without access to the final test cell, performance is reported at forecast horizons of 1, 3, 5 and 10 future assessments, and regular and randomized-redox regimes are examined separately.

The decision criterion is deliberately demanding: machine learning is accepted only when it improves upon persistence across cells without introducing severe held-out-cell failures.


In [ ]:
import os

# Apply the warning rule before parallel workers are created.
parallel_warning_rule = "ignore::UserWarning:sklearn.utils.parallel"
existing_warning_rules = os.environ.get("PYTHONWARNINGS", "")
if parallel_warning_rule not in existing_warning_rules.split(","):
    os.environ["PYTHONWARNINGS"] = ",".join(
        rule for rule in (existing_warning_rules, parallel_warning_rule) if rule
    )

import json
import sys
import warnings
from pathlib import Path
from time import perf_counter

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

from sofc_health.models.ml import add_lag_features
from sofc_health.validation.splits import leave_one_cell_out

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module=r"sklearn\.utils\.parallel",
)
warnings.filterwarnings(
    "ignore",
    message=r".*sklearn\.utils\.parallel\.delayed.*should be used.*",
    category=UserWarning,
)

sns.set_theme(style="whitegrid", context="talk")


def find_project_root(start: Path) -> Path:
    """Find the nearest parent directory containing pyproject.toml."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root containing pyproject.toml. "
        "Start Jupyter Notebook from inside the project directory."
    )


ROOT = find_project_root(Path.cwd())
PROJECT_ROOT = ROOT
DATA_PATH = ROOT / "data" / "processed" / "modeling_table.parquet"
OUTPUT_DIRECTORY = ROOT / "reports" / "tables" / "nested_ml"
FINAL_OUTPUT_DIRECTORY = OUTPUT_DIRECTORY / "final_cross_cell"
RESULT_DIR = FINAL_OUTPUT_DIRECTORY
FIGURE_DIR = ROOT / "reports" / "figures" / "nested_ml"

for directory in (OUTPUT_DIRECTORY, FINAL_OUTPUT_DIRECTORY, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# False gives a fast, reproducible notebook run using validated saved results.
# Change to True only when the full nested experiment must be recomputed.
RECOMPUTE_FINAL = False
N_JOBS = -1
RANDOM_STATE = 42

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Modeling table not found:\n{DATA_PATH}\n\nRun the data pipeline before continuing."
    )

table = (
    pd.read_parquet(DATA_PATH).sort_values(["cell_id", "assessment_index"]).reset_index(drop=True)
)
table["regime"] = np.where(
    table["cell_id"].str.startswith("R"),
    "randomized",
    "regular",
)

TARGET = "soh_composite_pct"
required_columns = {"cell_id", "assessment_index", TARGET}
missing_columns = required_columns.difference(table.columns)
if missing_columns:
    raise KeyError(f"Required columns are missing: {sorted(missing_columns)}")

duplicate_assessments = table.duplicated(subset=["cell_id", "assessment_index"]).sum()
if duplicate_assessments:
    raise ValueError(f"Duplicate cell-assessments found: {duplicate_assessments}")
if table[TARGET].isna().any():
    raise ValueError("Composite SOH contains missing values.")

print("Python:", sys.version.split()[0])
print("Project root:", ROOT)
print("Modeling-table shape:", table.shape)
print("Cells:", sorted(table["cell_id"].unique()))
print("Target:", TARGET)
print("Recompute final experiment:", RECOMPUTE_FINAL)

display(
    table.groupby(["regime", "cell_id"])
    .agg(
        assessments=("assessment_index", "nunique"),
        initial_soh=(TARGET, "first"),
        final_soh=(TARGET, "last"),
    )
    .round(3)
)

## Experimental design and leakage controls

The independent engineering unit is the physical cell, not an individual signal row. Measurements from the same cell share manufacturing history, operating exposure and degradation state, so randomly splitting rows would allow cell-specific information to appear in both training and testing data.

For each forecast horizon $h$, the model predicts the change in health from assessment $k$ to assessment $k+h$:

$$
\Delta SOH_{c,k,h}
=
SOH_{c,k+h}-SOH_{c,k}
$$

The future SOH prediction is reconstructed as:

$$
\widehat{SOH}_{c,k+h}
=
SOH_{c,k}+\widehat{\Delta SOH}_{c,k,h}
$$

Only information available at or before assessment $k$ is permitted. Lagged features are calculated within each physical cell, while imputation and scaling are fitted inside the relevant training fold.

The final validation protocol has two levels:

1. **Outer leave-one-cell-out evaluation:** one complete physical cell is reserved for the final test.
2. **Inner leave-one-training-cell-out selection:** representation, model and hyperparameters are selected using only the remaining cells.

The inner score gives equal weight to the regular and randomized-redox regimes. Persistence is included as a candidate, allowing the selection procedure to reject machine learning when added complexity is not justified.

Because reliable operating-hour intervals are unavailable, every horizon is expressed in future assessments rather than hours, cycles or calendar time.


In [ ]:
BASE_FEATURES = [
    "eis_r_ohmic_ohm",
    "eis_polarization_proxy_ohm",
    "iv_max_power_w_cm2",
    "tr_performance_current_a_cm2",
]

HORIZONS = [1, 3, 5, 10]
LAGS = (1, 2, 3)

# Compatibility names used in later cells
horizons = HORIZONS
lags = LAGS
target = TARGET


missing_features = [feature for feature in BASE_FEATURES if feature not in table.columns]

if missing_features:
    raise KeyError(f"Required physical features are missing: {missing_features}")


def make_predictor_names(feature_columns):
    """
    Generate current, lagged, change and current-SOH
    predictor names.
    """
    current_features = list(feature_columns)

    lagged_features = [f"{feature}_lag{lag}" for feature in feature_columns for lag in LAGS]

    change_features = [f"{feature}_delta1" for feature in feature_columns]

    return current_features + lagged_features + change_features + ["soh_at_origin_pct"]


def build_forecast_table(
    source,
    feature_columns,
):
    """
    Create causal lag features and direct
    multi-horizon forecast targets.
    """
    lagged = add_lag_features(
        source,
        feature_columns,
        lags=LAGS,
    )

    lagged.loc[:, "soh_at_origin_pct"] = lagged[TARGET]

    horizon_frames = []

    for horizon in HORIZONS:
        frame = lagged.copy()

        grouped = frame.groupby(
            "cell_id",
            sort=False,
        )

        frame.loc[:, "horizon"] = horizon

        frame.loc[
            :,
            "future_assessment_index",
        ] = grouped["assessment_index"].shift(-horizon)

        frame.loc[
            :,
            "future_soh_pct",
        ] = grouped[TARGET].shift(-horizon)

        # Make an explicit independent copy after filtering.
        # This prevents SettingWithCopyWarning.
        frame = frame.dropna(subset=["future_soh_pct"]).copy()

        frame.loc[
            :,
            "delta_soh_pct",
        ] = frame["future_soh_pct"] - frame["soh_at_origin_pct"]

        horizon_frames.append(frame)

    result = pd.concat(
        horizon_frames,
        ignore_index=True,
    )

    assessment_gap = result["future_assessment_index"] - result["assessment_index"]

    if not assessment_gap.eq(result["horizon"]).all():
        raise ValueError("Future-target alignment crossed a physical-cell boundary.")

    reconstruction_error = (
        (result["soh_at_origin_pct"] + result["delta_soh_pct"] - result["future_soh_pct"])
        .abs()
        .max()
    )

    if reconstruction_error > 1e-10:
        raise ValueError("Future SOH could not be reconstructed from the delta target.")

    return result


# ---------------------------------------------------------
# Absolute physical-feature representation
# ---------------------------------------------------------

predictors = make_predictor_names(BASE_FEATURES)

forecast_data = build_forecast_table(
    table,
    BASE_FEATURES,
)


# ---------------------------------------------------------
# Baseline-relative physical-feature representation
# ---------------------------------------------------------

relative_table = table.copy()
relative_features = []

for feature in BASE_FEATURES:
    baseline = relative_table.groupby("cell_id")[feature].transform("first")

    near_zero_baseline = baseline.abs() < 1e-12

    if near_zero_baseline.any():
        affected_cells = (
            relative_table.loc[
                near_zero_baseline,
                "cell_id",
            ]
            .drop_duplicates()
            .tolist()
        )

        raise ValueError(f"Near-zero baseline found for {feature}: {affected_cells}")

    relative_feature = f"{feature}_relative_pct"

    relative_table.loc[
        :,
        relative_feature,
    ] = 100 * (relative_table[feature] / baseline - 1)

    relative_features.append(relative_feature)


# Every relative trajectory must start at zero.
initial_relative_values = (
    relative_table.sort_values(["cell_id", "assessment_index"])
    .groupby("cell_id")[relative_features]
    .first()
    .abs()
)

maximum_initial_deviation = initial_relative_values.to_numpy().max()

if maximum_initial_deviation > 1e-10:
    raise ValueError("Relative feature trajectories do not begin at zero.")


relative_predictors = make_predictor_names(relative_features)

relative_forecast_data = build_forecast_table(
    relative_table,
    relative_features,
)


# ---------------------------------------------------------
# Representation registry used during nested validation
# ---------------------------------------------------------

REPRESENTATION_DATA = {
    "absolute": {
        "data": forecast_data,
        "predictors": predictors,
    },
    "relative": {
        "data": relative_forecast_data,
        "predictors": relative_predictors,
    },
}


# ---------------------------------------------------------
# Leakage audit
# ---------------------------------------------------------

forbidden_predictors = {
    "future_soh_pct",
    "future_assessment_index",
    "delta_soh_pct",
}

for representation, specification in REPRESENTATION_DATA.items():
    predictor_overlap = forbidden_predictors.intersection(specification["predictors"])

    if predictor_overlap:
        raise ValueError(
            f"Future information found in {representation} predictors: {sorted(predictor_overlap)}"
        )


# ---------------------------------------------------------
# Forecast-table audit
# ---------------------------------------------------------

feature_audit_rows = []

for representation, specification in REPRESENTATION_DATA.items():
    forecasting_frame = specification["data"]
    predictor_columns = specification["predictors"]

    for horizon, horizon_frame in forecasting_frame.groupby(
        "horizon",
        sort=True,
    ):
        feature_audit_rows.append(
            {
                "representation": representation,
                "horizon": horizon,
                "forecast_rows": len(horizon_frame),
                "cells": horizon_frame["cell_id"].nunique(),
                "predictors": len(predictor_columns),
                "missing_predictor_values": int(
                    horizon_frame[predictor_columns].isna().sum().sum()
                ),
            }
        )


feature_audit = pd.DataFrame(feature_audit_rows)

print("Causal forecast tables constructed.")
print(
    "Maximum initial relative deviation:",
    maximum_initial_deviation,
)
print(
    "Absolute forecast rows:",
    len(forecast_data),
)
print(
    "Relative forecast rows:",
    len(relative_forecast_data),
)
print(
    "Absolute predictors:",
    len(predictors),
)
print(
    "Relative predictors:",
    len(relative_predictors),
)

display(feature_audit)

In [ ]:
fold_records = []

for horizon in HORIZONS:
    horizon_data = forecast_data.loc[forecast_data["horizon"] == horizon].copy()

    for fold in leave_one_cell_out(horizon_data):
        train = horizon_data.loc[fold.train_indices]
        test = horizon_data.loc[fold.test_indices]
        train_cells = set(train["cell_id"])
        test_cells = set(test["cell_id"])
        overlap = train_cells.intersection(test_cells)

        if len(test_cells) != 1 or overlap:
            raise ValueError(f"Invalid outer fold: {fold.name}")

        fold_records.append(
            {
                "horizon": horizon,
                "held_out_cell": next(iter(test_cells)),
                "regime": test["regime"].iloc[0],
                "training_cells": len(train_cells),
                "training_rows": len(train),
                "test_rows": len(test),
                "cell_overlap": len(overlap),
            }
        )

fold_manifest = pd.DataFrame(fold_records)
expected_outer_folds = len(HORIZONS) * table["cell_id"].nunique()

assert len(fold_manifest) == expected_outer_folds
assert fold_manifest["cell_overlap"].eq(0).all()

print("Outer folds:", len(fold_manifest))
print("Expected folds:", expected_outer_folds)
print("Cell-leakage violations:", fold_manifest["cell_overlap"].sum())

display(
    fold_manifest.pivot(
        index="held_out_cell",
        columns="horizon",
        values="test_rows",
    )
)

## Candidate models and selection objective

Three model families are deliberately compared with persistence:

- **Ridge regression:** a regularized linear model that tests whether degradation changes combine approximately linearly.
- **Extra Trees:** a nonlinear tree ensemble that can capture interactions without requiring feature scaling.
- **Histogram gradient boosting:** a sequential nonlinear learner that fits residual structure stage by stage.

The small hyperparameter grids reflect the limited number of independent cells. A large search would increase selection variance without creating new experimental information.

For every candidate, MAE is calculated within each inner validation cell. The regular-cell MAE and randomized-cell MAE are then averaged equally:

$$
MAE_{balanced}
=
\frac{1}{2}
\left(
MAE_{regular}+MAE_{randomized}
\right)
$$

When only one regime is available in a particular inner comparison, the score uses the available regime and the limitation remains explicit in the reported table.


In [ ]:
MODEL_GRIDS = {
    "ridge": {
        "alpha": [0.1, 1.0, 10.0, 100.0],
    },
    "extra_trees": {
        "max_features": [0.5, 1.0],
        "min_samples_leaf": [2, 5],
    },
    "hist_gradient_boosting": {
        "learning_rate": [0.03, 0.07],
        "max_leaf_nodes": [7, 15],
    },
}


def build_estimator(model_name, parameters):
    """Build a pipeline whose preprocessing is fitted inside each fold."""
    if model_name == "ridge":
        return Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(strategy="median", add_indicator=True),
                ),
                ("scaler", RobustScaler()),
                ("regressor", Ridge(alpha=parameters["alpha"])),
            ]
        )

    if model_name == "extra_trees":
        return Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(strategy="median", add_indicator=True),
                ),
                (
                    "regressor",
                    ExtraTreesRegressor(
                        n_estimators=300,
                        max_features=parameters["max_features"],
                        min_samples_leaf=parameters["min_samples_leaf"],
                        random_state=RANDOM_STATE,
                        n_jobs=N_JOBS,
                    ),
                ),
            ]
        )

    if model_name == "hist_gradient_boosting":
        return Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(strategy="median", add_indicator=True),
                ),
                (
                    "regressor",
                    HistGradientBoostingRegressor(
                        learning_rate=parameters["learning_rate"],
                        max_leaf_nodes=parameters["max_leaf_nodes"],
                        max_iter=300,
                        l2_regularization=1.0,
                        early_stopping=False,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        )

    raise ValueError(f"Unknown model: {model_name}")


def calculate_regression_metrics(actual, predicted):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)
    signed_error = predicted - actual
    return {
        "mae": mean_absolute_error(actual, predicted),
        "rmse": np.sqrt(mean_squared_error(actual, predicted)),
        "bias": np.mean(signed_error),
        "r2": r2_score(actual, predicted),
    }


search_space_summary = pd.DataFrame(
    [
        {
            "model": model_name,
            "candidate_configurations": len(list(ParameterGrid(grid))),
            "configurations": list(ParameterGrid(grid)),
        }
        for model_name, grid in MODEL_GRIDS.items()
    ]
)

display(search_space_summary)
print(
    "ML candidate configurations per representation:",
    search_space_summary["candidate_configurations"].sum(),
)

In [ ]:
def tune_with_balanced_inner_loco(outer_held_out_cell, horizon):
    """Select representation, model and parameters without using the test cell."""
    candidate_records = []

    for representation, specification in REPRESENTATION_DATA.items():
        predictor_columns = specification["predictors"]
        horizon_data = specification["data"].loc[specification["data"]["horizon"] == horizon]
        outer_training_data = horizon_data.loc[horizon_data["cell_id"] != outer_held_out_cell]
        inner_validation_cells = sorted(outer_training_data["cell_id"].unique())

        for model_name, grid in MODEL_GRIDS.items():
            for parameters in ParameterGrid(grid):
                cell_records = []

                for inner_validation_cell in inner_validation_cells:
                    inner_train = outer_training_data.loc[
                        outer_training_data["cell_id"] != inner_validation_cell
                    ]
                    inner_validation = outer_training_data.loc[
                        outer_training_data["cell_id"] == inner_validation_cell
                    ]

                    estimator = build_estimator(model_name, parameters)
                    estimator.fit(
                        inner_train[predictor_columns],
                        inner_train["delta_soh_pct"],
                    )
                    predicted_delta = estimator.predict(inner_validation[predictor_columns])
                    actual_delta = inner_validation["delta_soh_pct"].to_numpy()

                    cell_records.append(
                        {
                            "cell_id": inner_validation_cell,
                            "regime": inner_validation["regime"].iloc[0],
                            "mae": mean_absolute_error(
                                actual_delta,
                                predicted_delta,
                            ),
                            "bias": np.mean(predicted_delta - actual_delta),
                        }
                    )

                cell_scores = pd.DataFrame(cell_records)
                regime_scores = cell_scores.groupby("regime")["mae"].mean()

                candidate_records.append(
                    {
                        "outer_held_out_cell": outer_held_out_cell,
                        "horizon": horizon,
                        "representation": representation,
                        "model": model_name,
                        "parameters": dict(parameters),
                        "inner_balanced_mae": regime_scores.dropna().mean(),
                        "inner_regular_mae": regime_scores.get(
                            "regular",
                            np.nan,
                        ),
                        "inner_randomized_mae": regime_scores.get(
                            "randomized",
                            np.nan,
                        ),
                        "inner_macro_cell_mae": cell_scores["mae"].mean(),
                        "inner_worst_cell_mae": cell_scores["mae"].max(),
                        "inner_macro_bias": cell_scores["bias"].mean(),
                    }
                )

    return (
        pd.DataFrame(candidate_records)
        .sort_values(["inner_balanced_mae", "inner_worst_cell_mae"])
        .reset_index(drop=True)
    )


def build_persistence_candidate(outer_held_out_cell, horizon):
    """Score persistence using the same inner-cell and regime weighting."""
    horizon_data = forecast_data.loc[forecast_data["horizon"] == horizon]
    outer_training_data = horizon_data.loc[horizon_data["cell_id"] != outer_held_out_cell]
    cell_records = []

    for inner_validation_cell in sorted(outer_training_data["cell_id"].unique()):
        inner_validation = outer_training_data.loc[
            outer_training_data["cell_id"] == inner_validation_cell
        ]
        actual_delta = inner_validation["delta_soh_pct"].to_numpy()
        cell_records.append(
            {
                "cell_id": inner_validation_cell,
                "regime": inner_validation["regime"].iloc[0],
                "mae": mean_absolute_error(
                    actual_delta,
                    np.zeros(len(inner_validation)),
                ),
                "bias": np.mean(-actual_delta),
            }
        )

    cell_scores = pd.DataFrame(cell_records)
    regime_scores = cell_scores.groupby("regime")["mae"].mean()

    return {
        "outer_held_out_cell": outer_held_out_cell,
        "horizon": horizon,
        "representation": "none",
        "model": "persistence",
        "parameters": {},
        "inner_balanced_mae": regime_scores.dropna().mean(),
        "inner_regular_mae": regime_scores.get("regular", np.nan),
        "inner_randomized_mae": regime_scores.get("randomized", np.nan),
        "inner_macro_cell_mae": cell_scores["mae"].mean(),
        "inner_worst_cell_mae": cell_scores["mae"].max(),
        "inner_macro_bias": cell_scores["bias"].mean(),
    }


print("Nested cross-cell selection utilities are ready.")

In [ ]:
def run_final_nested_experiment():
    """Run the confirmatory 32-fold experiment and save checkpoints."""
    metric_records = []
    prediction_frames = []
    tuning_frames = []
    start_time = perf_counter()
    total_folds = len(HORIZONS) * table["cell_id"].nunique()
    completed_folds = 0

    for horizon in HORIZONS:
        for held_out_cell in sorted(table["cell_id"].unique()):
            candidate_tuning = tune_with_balanced_inner_loco(
                outer_held_out_cell=held_out_cell,
                horizon=horizon,
            )
            candidate_tuning = pd.concat(
                [
                    candidate_tuning,
                    pd.DataFrame([build_persistence_candidate(held_out_cell, horizon)]),
                ],
                ignore_index=True,
            )
            candidate_tuning = candidate_tuning.sort_values(
                ["inner_balanced_mae", "inner_worst_cell_mae"]
            ).reset_index(drop=True)
            candidate_tuning["rank"] = np.arange(len(candidate_tuning)) + 1
            candidate_tuning["selected"] = candidate_tuning["rank"].eq(1)
            best_candidate = candidate_tuning.iloc[0]

            selected_model = best_candidate["model"]
            selected_representation = best_candidate["representation"]
            selected_parameters = best_candidate["parameters"]

            if selected_model == "persistence":
                specification = REPRESENTATION_DATA["absolute"]
                selected_predictors = []
            else:
                specification = REPRESENTATION_DATA[selected_representation]
                selected_predictors = specification["predictors"]

            horizon_data = specification["data"].loc[specification["data"]["horizon"] == horizon]
            outer_train = horizon_data.loc[horizon_data["cell_id"] != held_out_cell]
            outer_test = horizon_data.loc[horizon_data["cell_id"] == held_out_cell]

            actual_delta = outer_test["delta_soh_pct"].to_numpy()
            origin_soh = outer_test["soh_at_origin_pct"].to_numpy()
            actual_soh = outer_test["future_soh_pct"].to_numpy()

            if selected_model == "persistence":
                predicted_delta = np.zeros(len(outer_test))
            else:
                estimator = build_estimator(
                    selected_model,
                    selected_parameters,
                )
                estimator.fit(
                    outer_train[selected_predictors],
                    outer_train["delta_soh_pct"],
                )
                predicted_delta = estimator.predict(outer_test[selected_predictors])

            predicted_soh = origin_soh + predicted_delta
            persistence_prediction = origin_soh.copy()
            persistence_mae = mean_absolute_error(
                actual_soh,
                persistence_prediction,
            )
            metrics = calculate_regression_metrics(actual_soh, predicted_soh)
            mae_skill = np.nan if persistence_mae == 0 else 1 - metrics["mae"] / persistence_mae

            metric_records.append(
                {
                    "horizon": horizon,
                    "cell_id": held_out_cell,
                    "regime": outer_test["regime"].iloc[0],
                    "selected_model": selected_model,
                    "selected_representation": selected_representation,
                    "selected_parameters": json.dumps(
                        selected_parameters,
                        sort_keys=True,
                    ),
                    "inner_balanced_mae": best_candidate["inner_balanced_mae"],
                    "inner_regular_mae": best_candidate["inner_regular_mae"],
                    "inner_randomized_mae": best_candidate["inner_randomized_mae"],
                    "inner_worst_cell_mae": best_candidate["inner_worst_cell_mae"],
                    **metrics,
                    "persistence_mae": persistence_mae,
                    "mae_skill": mae_skill,
                }
            )

            prediction_frame = outer_test[
                [
                    "cell_id",
                    "regime",
                    "assessment_index",
                    "future_assessment_index",
                ]
            ].copy()
            prediction_frame["horizon"] = horizon
            prediction_frame["selected_model"] = selected_model
            prediction_frame["selected_representation"] = selected_representation
            prediction_frame["actual_delta_soh_pct"] = actual_delta
            prediction_frame["predicted_delta_soh_pct"] = predicted_delta
            prediction_frame["actual_future_soh_pct"] = actual_soh
            prediction_frame["predicted_future_soh_pct"] = predicted_soh
            prediction_frame["persistence_prediction_pct"] = persistence_prediction
            prediction_frame["signed_error_pct"] = predicted_soh - actual_soh
            prediction_frames.append(prediction_frame)

            tuning_export = candidate_tuning.copy()
            tuning_export["parameters"] = tuning_export["parameters"].apply(
                lambda value: json.dumps(value, sort_keys=True)
            )
            tuning_frames.append(tuning_export)

            completed_folds += 1
            elapsed_minutes = (perf_counter() - start_time) / 60
            print(
                f"[{completed_folds:02d}/{total_folds}] "
                f"horizon={horizon}, held_out={held_out_cell}, "
                f"selected={selected_model}/{selected_representation}, "
                f"elapsed={elapsed_minutes:.1f} min"
            )

            pd.DataFrame(metric_records).to_csv(
                FINAL_OUTPUT_DIRECTORY / "metrics_checkpoint.csv",
                index=False,
            )
            pd.concat(prediction_frames, ignore_index=True).to_csv(
                FINAL_OUTPUT_DIRECTORY / "predictions_checkpoint.csv",
                index=False,
            )
            pd.concat(tuning_frames, ignore_index=True).to_csv(
                FINAL_OUTPUT_DIRECTORY / "tuning_checkpoint.csv",
                index=False,
            )

    final_metrics = pd.DataFrame(metric_records)
    final_predictions = pd.concat(prediction_frames, ignore_index=True)
    final_tuning = pd.concat(tuning_frames, ignore_index=True)

    final_metrics.to_csv(
        FINAL_OUTPUT_DIRECTORY / "final_selected_metrics.csv",
        index=False,
    )
    final_predictions.to_csv(
        FINAL_OUTPUT_DIRECTORY / "final_selected_predictions.csv",
        index=False,
    )
    final_tuning.to_csv(
        FINAL_OUTPUT_DIRECTORY / "final_nested_tuning.csv",
        index=False,
    )

    print(f"Final experiment completed in {(perf_counter() - start_time) / 60:.1f} minutes.")
    return final_metrics, final_predictions, final_tuning


print("Full recomputation function defined. No experiment has been started.")

## Reproducible execution policy

The confirmatory nested experiment is computationally expensive, but its validated fold-level results are stored as CSV files. The default notebook path loads those results and reproduces every summary and figure without repeating model selection.

Set `RECOMPUTE_FINAL = True` in the setup cell only when a complete retraining run is intentionally required. With the default value of `False`, a clean **Restart and Run All** remains fast, deterministic and suitable for GitHub review.


In [ ]:
RESULT_FILES = {
    "metrics": FINAL_OUTPUT_DIRECTORY / "final_selected_metrics.csv",
    "predictions": FINAL_OUTPUT_DIRECTORY / "final_selected_predictions.csv",
    "tuning": FINAL_OUTPUT_DIRECTORY / "final_nested_tuning.csv",
}

if RECOMPUTE_FINAL:
    (
        final_selected_metrics,
        final_selected_predictions,
        final_nested_tuning,
    ) = run_final_nested_experiment()
else:
    missing_result_files = [path for path in RESULT_FILES.values() if not path.exists()]
    if missing_result_files:
        missing_text = "\n".join(f" - {path}" for path in missing_result_files)
        raise FileNotFoundError(
            "Validated final results are missing:\n"
            f"{missing_text}\n\n"
            "Set RECOMPUTE_FINAL = True to create them."
        )

    final_selected_metrics = pd.read_csv(RESULT_FILES["metrics"])
    final_selected_predictions = pd.read_csv(RESULT_FILES["predictions"])
    final_nested_tuning = pd.read_csv(RESULT_FILES["tuning"])

expected_counts = {
    "metrics": 32,
    "predictions": 1520,
    "tuning": 800,
}
actual_counts = {
    "metrics": len(final_selected_metrics),
    "predictions": len(final_selected_predictions),
    "tuning": len(final_nested_tuning),
}

if actual_counts != expected_counts:
    raise ValueError(
        f"Final-result audit failed. Expected {expected_counts}, received {actual_counts}."
    )
if final_selected_metrics.duplicated(["cell_id", "horizon"]).any():
    raise ValueError("Duplicate cell-horizon metric rows were found.")
if final_selected_predictions.duplicated(["cell_id", "horizon", "assessment_index"]).any():
    raise ValueError("Duplicate final prediction rows were found.")

print("Final nested-evaluation audit passed.")
print("Metric rows:", actual_counts["metrics"])
print("Prediction rows:", actual_counts["predictions"])
print("Tuning rows:", actual_counts["tuning"])
print("Execution mode:", "recomputed" if RECOMPUTE_FINAL else "cached results")

## Final performance assessment

The primary score measures MAE improvement relative to persistence:

$$
Skill_{c,h}
=
1-
\frac{MAE_{c,h}^{selected}}{MAE_{c,h}^{persistence}}
$$

- $Skill>0$: the selected model improves upon persistence.
- $Skill=0$: no improvement.
- $Skill<0$: the selected model increases error.

Metrics are calculated for each held-out cell first and then averaged equally across cells. This prevents the longest cell trajectory from dominating the engineering conclusion.

A catastrophic failure is defined as $Skill<-1$, meaning that the selected model has more than twice the MAE of persistence. A horizon passes the acceptance gate only when macro skill and median cell skill are positive and no catastrophic failure occurs.


In [ ]:
def summarize_outer_results(group):
    model_macro_mae = group["mae"].mean()
    persistence_macro_mae = group["persistence_mae"].mean()
    return pd.Series(
        {
            "cells": group["cell_id"].nunique(),
            "model_macro_mae": model_macro_mae,
            "persistence_macro_mae": persistence_macro_mae,
            "macro_mae_skill": 1 - model_macro_mae / persistence_macro_mae,
            "median_cell_skill": group["mae_skill"].median(),
            "minimum_cell_skill": group["mae_skill"].min(),
            "maximum_cell_skill": group["mae_skill"].max(),
            "cells_beating_persistence": int((group["mae_skill"] > 0).sum()),
            "catastrophic_failures": int((group["mae_skill"] < -1).sum()),
            "mean_bias_pct": group["bias"].mean(),
            "mean_absolute_bias_pct": group["bias"].abs().mean(),
        }
    )


horizon_summary = (
    final_selected_metrics.groupby("horizon", sort=True)
    .apply(summarize_outer_results, include_groups=False)
    .reset_index()
)
regime_summary = (
    final_selected_metrics.groupby(["horizon", "regime"], sort=True)
    .apply(summarize_outer_results, include_groups=False)
    .reset_index()
)
selection_counts = (
    final_selected_metrics.groupby(
        ["horizon", "selected_model", "selected_representation"],
        dropna=False,
    )
    .size()
    .rename("selected_folds")
    .reset_index()
    .sort_values(["horizon", "selected_folds"], ascending=[True, False])
)

horizon_summary["positive_macro_skill"] = horizon_summary["macro_mae_skill"] > 0
horizon_summary["positive_median_skill"] = horizon_summary["median_cell_skill"] > 0
horizon_summary["no_catastrophic_failure"] = horizon_summary["catastrophic_failures"] == 0
horizon_summary["passes_acceptance_gate"] = (
    horizon_summary["positive_macro_skill"]
    & horizon_summary["positive_median_skill"]
    & horizon_summary["no_catastrophic_failure"]
)

horizon_summary.to_csv(
    FINAL_OUTPUT_DIRECTORY / "final_horizon_summary.csv",
    index=False,
)
regime_summary.to_csv(
    FINAL_OUTPUT_DIRECTORY / "final_regime_summary.csv",
    index=False,
)
selection_counts.to_csv(
    FINAL_OUTPUT_DIRECTORY / "final_selection_counts.csv",
    index=False,
)

print("Final horizon-level performance")
display(horizon_summary.round(3))
print("Performance by redox regime")
display(regime_summary.round(3))
print("Selected model and representation counts")
display(selection_counts)

In [ ]:
fold_diagnostics = final_selected_metrics[
    [
        "horizon",
        "cell_id",
        "regime",
        "selected_model",
        "selected_representation",
        "mae",
        "persistence_mae",
        "mae_skill",
        "bias",
        "rmse",
        "r2",
    ]
].copy()
fold_diagnostics["mae_change_vs_persistence_pct"] = 100 * (
    fold_diagnostics["mae"] / fold_diagnostics["persistence_mae"] - 1
)
fold_diagnostics["outcome"] = np.select(
    [
        fold_diagnostics["mae_skill"] < -1,
        fold_diagnostics["mae_skill"] > 0,
        fold_diagnostics["mae_skill"].abs() < 1e-12,
    ],
    ["catastrophic failure", "ML improvement", "persistence selected"],
    default="ML worse",
)
fold_diagnostics = fold_diagnostics.sort_values(
    ["horizon", "mae_skill"],
    ascending=[True, False],
).reset_index(drop=True)
fold_diagnostics.to_csv(
    FINAL_OUTPUT_DIRECTORY / "final_cell_horizon_diagnostics.csv",
    index=False,
)

print("Largest improvements")
display(
    fold_diagnostics.nlargest(5, "mae_skill")[
        [
            "horizon",
            "cell_id",
            "regime",
            "selected_model",
            "selected_representation",
            "mae_skill",
            "mae_change_vs_persistence_pct",
        ]
    ].round(3)
)
print("Largest failures")
display(
    fold_diagnostics.nsmallest(5, "mae_skill")[
        [
            "horizon",
            "cell_id",
            "regime",
            "selected_model",
            "selected_representation",
            "mae",
            "persistence_mae",
            "mae_skill",
            "bias",
        ]
    ].round(3)
)
print("Unrounded horizon-level skill")
for row in horizon_summary.itertuples():
    print(f"h = {int(row.horizon):>2}: {row.macro_mae_skill:.8f}")

In [ ]:
def calculate_selected_fold_ood(metric_row):
    representation = str(metric_row.selected_representation)
    if representation not in REPRESENTATION_DATA:
        return {
            "outside_minmax_rate": np.nan,
            "outside_robust_rate": np.nan,
            "shifted_predictors": 0,
        }

    specification = REPRESENTATION_DATA[representation]
    frame = specification["data"]
    predictor_columns = specification["predictors"]
    horizon_data = frame.loc[frame["horizon"] == metric_row.horizon]
    train = horizon_data.loc[horizon_data["cell_id"] != metric_row.cell_id]
    test = horizon_data.loc[horizon_data["cell_id"] == metric_row.cell_id]

    total_values = 0
    outside_minmax = 0
    outside_robust = 0
    shifted_predictors = 0

    for feature in predictor_columns:
        train_values = pd.to_numeric(train[feature], errors="coerce").dropna()
        test_values = pd.to_numeric(test[feature], errors="coerce").dropna()
        if train_values.empty or test_values.empty:
            continue

        minimum = train_values.min()
        maximum = train_values.max()
        lower = train_values.quantile(0.01)
        upper = train_values.quantile(0.99)
        feature_minmax = (test_values.lt(minimum) | test_values.gt(maximum)).sum()
        feature_robust = (test_values.lt(lower) | test_values.gt(upper)).sum()

        total_values += len(test_values)
        outside_minmax += int(feature_minmax)
        outside_robust += int(feature_robust)
        shifted_predictors += int(feature_robust > 0)

    return {
        "outside_minmax_rate": outside_minmax / total_values,
        "outside_robust_rate": outside_robust / total_values,
        "shifted_predictors": shifted_predictors,
    }


ood_records = []
for metric_row in final_selected_metrics.itertuples(index=False):
    ood_records.append(
        {
            "horizon": metric_row.horizon,
            "cell_id": metric_row.cell_id,
            **calculate_selected_fold_ood(metric_row),
        }
    )

selected_fold_ood = pd.DataFrame(ood_records)
failure_audit = fold_diagnostics.merge(
    selected_fold_ood,
    on=["horizon", "cell_id"],
    how="left",
)
failure_audit.to_csv(
    FINAL_OUTPUT_DIRECTORY / "final_failure_and_ood_audit.csv",
    index=False,
)

print("Distribution shift in the largest failures")
display(
    failure_audit.nsmallest(8, "mae_skill")[
        [
            "horizon",
            "cell_id",
            "regime",
            "selected_model",
            "selected_representation",
            "mae_skill",
            "outside_minmax_rate",
            "outside_robust_rate",
            "shifted_predictors",
        ]
    ].round(3)
)

In [ ]:
sns.set_theme(style="white", context="talk")

cell_order = ["N1", "N2", "N3", "N4", "N5", "N6", "R1", "R2"]
horizon_order = [1, 3, 5, 10]
skill_matrix = final_selected_metrics.pivot(
    index="cell_id",
    columns="horizon",
    values="mae_skill",
).reindex(index=cell_order, columns=horizon_order)

model_codes = {
    "persistence": "P",
    "ridge": "Ridge",
    "extra_trees": "ET",
    "hist_gradient_boosting": "HGB",
}
representation_codes = {"absolute": "A", "relative": "R", "none": ""}
annotation_matrix = pd.DataFrame("", index=cell_order, columns=horizon_order)

for row in final_selected_metrics.itertuples():
    model_code = model_codes[row.selected_model]
    representation_code = representation_codes.get(
        str(row.selected_representation),
        "",
    )
    method_label = model_code if model_code == "P" else f"{model_code}-{representation_code}"
    annotation_matrix.loc[row.cell_id, row.horizon] = f"{row.mae_skill:+.2f}\n{method_label}"

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(
    skill_matrix,
    annot=annotation_matrix,
    fmt="",
    cmap="RdYlGn",
    center=0,
    vmin=-2.0,
    vmax=0.6,
    linewidths=1.2,
    linecolor="white",
    cbar_kws={"label": "MAE skill relative to persistence", "shrink": 0.82},
    annot_kws={"fontsize": 10, "fontweight": "bold"},
    ax=ax,
)
ax.axhline(6, color="black", linewidth=2.0)

for row_index, cell_id in enumerate(cell_order):
    for column_index, horizon in enumerate(horizon_order):
        if skill_matrix.loc[cell_id, horizon] < -1:
            ax.add_patch(
                patches.Rectangle(
                    (column_index, row_index),
                    1,
                    1,
                    fill=False,
                    edgecolor="black",
                    linewidth=3,
                )
            )

ax.set_title(
    "Cross-cell forecast skill by cell and horizon\n"
    "Positive values indicate lower MAE than persistence",
    fontsize=16,
    fontweight="bold",
    pad=18,
)
ax.set_xlabel("Forecast horizon, future assessments")
ax.set_ylabel("Held-out physical cell")
ax.set_xticklabels([f"h = {horizon}" for horizon in horizon_order], rotation=0)
ax.set_yticklabels(cell_order, rotation=0)

fig.text(
    0.5,
    0.02,
    "P: persistence | ET: Extra Trees | HGB: histogram gradient boosting | "
    "A: absolute | R: baseline-relative\n"
    "Black borders identify cases with more than twice the persistence MAE.",
    ha="center",
    fontsize=9.5,
    color="dimgray",
)
plt.tight_layout(rect=[0, 0.09, 1, 1])

skill_figure_path = FIGURE_DIR / "final_cell_horizon_skill_heatmap.png"
plt.savefig(skill_figure_path, dpi=300, bbox_inches="tight")
plt.show()
print("Figure saved to:", skill_figure_path)

In [ ]:
sns.set_theme(style="whitegrid", context="talk")

diagnostic_cases = [
    {
        "cell_id": "N2",
        "horizon": 10,
        "case_label": "Strongest successful transfer",
    },
    {
        "cell_id": "R2",
        "horizon": 10,
        "case_label": "Largest cross-cell failure",
    },
]
model_display_names = {
    "ridge": "Ridge",
    "extra_trees": "Extra Trees",
    "hist_gradient_boosting": "Histogram gradient boosting",
    "persistence": "Persistence",
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)

for ax, case in zip(axes, diagnostic_cases, strict=True):
    cell_id = case["cell_id"]
    horizon = case["horizon"]
    predictions = (
        final_selected_predictions.loc[
            (final_selected_predictions["cell_id"] == cell_id)
            & (final_selected_predictions["horizon"] == horizon)
        ]
        .sort_values("future_assessment_index")
        .copy()
    )
    metric = final_selected_metrics.loc[
        (final_selected_metrics["cell_id"] == cell_id)
        & (final_selected_metrics["horizon"] == horizon)
    ].iloc[0]
    if predictions.empty:
        raise ValueError(f"No predictions found for {cell_id}, h = {horizon}.")

    selected_model = model_display_names[metric["selected_model"]]
    representation = str(metric["selected_representation"]).capitalize()

    ax.plot(
        predictions["future_assessment_index"],
        predictions["actual_future_soh_pct"],
        color="black",
        linewidth=2.5,
        marker="o",
        markersize=4,
        label="Observed future SOH",
        zorder=3,
    )
    ax.plot(
        predictions["future_assessment_index"],
        predictions["predicted_future_soh_pct"],
        color="#0072B2",
        linewidth=2.2,
        label=f"{selected_model} prediction",
        zorder=2,
    )
    ax.plot(
        predictions["future_assessment_index"],
        predictions["persistence_prediction_pct"],
        color="#7F7F7F",
        linewidth=2.0,
        linestyle="--",
        label="Persistence prediction",
        zorder=1,
    )
    ax.set_title(
        f"{case['case_label']}\nCell {cell_id}, h = {horizon}",
        fontsize=15,
        fontweight="bold",
    )
    ax.set_xlabel("Future assessment index")
    ax.set_ylabel("Composite SOH (%)")
    ax.text(
        0.03,
        0.04,
        (
            f"Selected: {selected_model}, {representation.lower()} features\n"
            f"MAE: {metric['mae']:.2f}% | "
            f"Persistence MAE: {metric['persistence_mae']:.2f}%\n"
            f"MAE skill: {metric['mae_skill']:+.2f}"
        ),
        transform=ax.transAxes,
        fontsize=10,
        verticalalignment="bottom",
        bbox={
            "boxstyle": "round,pad=0.5",
            "facecolor": "white",
            "edgecolor": "lightgray",
            "alpha": 0.9,
        },
    )

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.98),
    ncol=3,
    frameon=False,
    fontsize=11,
)
fig.suptitle(
    "Why average performance is insufficient for deployment",
    fontsize=18,
    fontweight="bold",
    y=1.03,
)
plt.tight_layout(rect=[0, 0, 1, 0.88])

trajectory_path = FIGURE_DIR / "representative_success_and_failure_forecasts.png"
plt.savefig(trajectory_path, dpi=300, bbox_inches="tight")
plt.show()
print("Figure saved to:", trajectory_path)

## Engineering conclusion

### Primary result

The final confirmatory experiment does not support replacing persistence with the selected machine-learning system across unseen SOFC cells.

| Horizon | Selected macro MAE | Persistence macro MAE | Macro skill | Cells improved | Decision |
|---:|---:|---:|---:|---:|---|
| 1 | 0.669 | 0.669 | 0.000 | 4 of 8 | Practical tie |
| 3 | 1.493 | 1.304 | -0.145 | 1 of 8 | Persistence preferred |
| 5 | 1.976 | 1.612 | -0.226 | 1 of 8 | Persistence preferred |
| 10 | 2.537 | 2.259 | -0.123 | 5 of 8 | Promising but unsafe |

At horizon 1, the unrounded macro skill is approximately 0.000048, equivalent to only 0.005% improvement. Median cell skill is negative and one cell fails catastrophically, so this result is a practical tie rather than an ML victory.

At horizons 3 and 5, inner validation selects persistence for five of the eight outer folds. This is evidence that the available multimodal features do not add sufficiently transferable information at these horizons.

At horizon 10, the selected system improves five of eight cells and achieves a positive median cell skill of 0.132. However, one severe failure makes overall MAE 12.3% worse than persistence. Median performance therefore conceals unacceptable deployment tail risk.

### Successful transfer and failure analysis

The strongest success is regular-redox cell N2 at horizon 10. Baseline-relative Extra Trees reduces MAE from 2.525 to 1.145 SOH percentage points, a 54.7% improvement over persistence. This shows that normalized degradation trajectories can contain useful long-horizon information when the test cell follows patterns represented in training.

The largest failure is randomized-redox cell R2 at horizon 10. Histogram gradient boosting increases MAE from 1.986 to 5.955 and produces a bias of +5.955 SOH percentage points. This is systematic health overprediction, not an isolated random miss.

Additional catastrophic failures occur for N6 at horizon 1 and N3 at horizons 3 and 5. The distribution-shift audit indicates that some failures coincide with predictors outside the training range, while others occur despite limited observed covariate shift. The latter pattern is consistent with possible concept shift, but does not by itself prove a physical mechanism.

### Effect of redox regime

At horizon 10, regular cells show an 11.9% macro improvement, while randomized cells show a 50.7% deterioration. Separate reporting is therefore necessary. Only two randomized-redox cells are available, so their results provide important failure evidence but cannot establish general regime performance.

### Model decision

No horizon satisfies all three acceptance requirements:

1. Positive macro skill
2. Positive median cell skill
3. No catastrophic held-out-cell failure

Persistence remains the operational benchmark. Baseline-relative Extra Trees at horizon 10 is retained as a research candidate for regular-redox cells, not as a deployable replacement.

A defensible monitoring system would use persistence by default, screen every ML prediction for distribution shift, report predictive uncertainty, and fall back to persistence whenever transfer evidence is weak.

### Limitations

- Only eight independent physical cells are available.
- Only two cells use randomized redox cycling.
- The data come from laboratory redox experiments, not field-installed telemetry.
- Forecast horizons represent future assessments, not operating hours.
- Composite SOH requires an initial cell diagnostic as a cold-start baseline.
- No sustained 75% or 80% end-of-life event was observed.
- Exact remaining useful life cannot be validated from these trajectories.
- Baseline-relative feature design requires confirmation on an independent dataset.

### Final interpretation

The strongest result is not that a complex model wins. It is that apparent improvements are tested under genuine cross-cell transfer and rejected when they are unstable. The analysis identifies a promising long-horizon signal for several regular cells while exposing severe risk under cell and regime shift.

Notebook 07 will build on this result by quantifying predictive uncertainty and formalizing fallback behaviour for censored lifetime data.


## Reproducibility and generated artifacts

The final nested cross-cell experiment produces the following validated tables in `reports/tables/nested_ml/final_cross_cell/`:

- `final_selected_metrics.csv`: performance for each held-out cell and horizon
- `final_selected_predictions.csv`: observation-level forecasts
- `final_nested_tuning.csv`: inner-validation and model-selection results
- `final_horizon_summary.csv`: performance aggregated by forecast horizon
- `final_regime_summary.csv`: separate regular and randomized-redox results
- `final_selection_counts.csv`: selected models and feature representations
- `final_cell_horizon_diagnostics.csv`: cell-level success and failure analysis
- `final_failure_and_ood_audit.csv`: distribution-shift diagnostics

Publication-ready figures are saved in `reports/figures/nested_ml/`.

For normal review and reproducibility, use:

`RECOMPUTE_FINAL = False`

This loads the validated results and regenerates the analysis without repeating the computationally expensive nested experiment.

Set `RECOMPUTE_FINAL = True` only when intentionally retraining every model and reproducing all 32 outer validation folds.